In [1]:
## Run this cell FIRST

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


spark = SparkSession.builder \
    .appName("Sedona") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "0") \
    .config("spark.dynamicAllocation.initialExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "4") \
    .config("spark.dynamicAllocation.executorIdleTimeout", "10s") \
    .getOrCreate()

@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("EJECUTANDO:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        result.show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/23 01:01:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/23 01:01:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/23 01:01:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/08/23 01:01:12 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive} is set, falling back to uploading libraries under SPARK_HOME.


In [10]:
from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.FileSystem")
java_import(spark._jvm, "org.apache.hadoop.fs.Path")

fs = spark._jvm.FileSystem.get(
    spark.sparkContext._jsc.hadoopConfiguration()
)

fs.delete(
    spark._jvm.Path("/spark-warehouse"),
    True
)

fs.mkdirs(
    spark._jvm.Path("/spark-warehouse")
)

print("spark-warehouse clean")


spark-warehouse clean


In [15]:
%%sql

-- create trips in delta

CREATE OR REPLACE TEMP VIEW citibike_csv
USING csv
OPTIONS (
  path 'file:///opt/data/2023-citibike-tripdata',
  header 'true',
  inferSchema 'true'
);

DROP TABLE IF EXISTS trips;
CREATE TABLE trips
USING DELTA
AS 
SELECT * 
FROM citibike_csv
WHERE start_lng IS NOT NULL AND start_lat IS NOT NULL AND end_lng IS NOT NULL AND end_lat IS NOT NULL ;

ALTER TABLE trips ADD COLUMN origin GEOMETRY;
ALTER TABLE trips ADD COLUMN target GEOMETRY;

-- For spark 3
-- ALTER TABLE trips ADD COLUMN duration_ms INTEGER;

-- For Spark 4, avoid Overflow
ALTER TABLE trips ADD COLUMN duration_ms BIGINT;


UPDATE trips SET 
  origin = st_setsrid(st_point(start_lng, start_lat), 4326),
  target = st_setsrid(st_point(end_lng, end_lat), 4326),
  duration_ms= (unix_timestamp(ended_at) - unix_timestamp(started_at)) * 1000;





>>> CREATE OR REPLACE TEMP VIEW citibike_csv
USING csv
OPTIONS (
  path 'file:///opt...


++
||
++
++


>>> DROP TABLE IF EXISTS trips...
++
||
++
++


>>> CREATE TABLE trips
USING DELTA
AS 
SELECT * 
FROM citibike_csv
WHERE start_lng I...


++
||
++
++


>>> ALTER TABLE trips ADD COLUMN origin GEOMETRY...
++
||
++
++


>>> ALTER TABLE trips ADD COLUMN target GEOMETRY...
++
||
++
++


>>> ALTER TABLE trips ADD COLUMN duration_ms BIGINT...
++
||
++
++


>>> UPDATE trips SET 
  origin = st_setsrid(st_point(start_lng, start_lat), 4326),
 ...


26/08/22 22:54:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/22 22:56:46 WARN UpdateCommand: Could not validate number of records due to missing statistics.


+-----------------+
|num_affected_rows|
+-----------------+
|35002469         |
+-----------------+



In [16]:
%%sql

-- o copiar en HDFS
-- hdfs dfs -mkdir -p /opt/data
-- hdfs dfs -put /opt/data/nynta2020_26b /opt/data/nynta2020_26b

CREATE OR REPLACE TEMP VIEW auxi_nta_2020 USING shapefile
OPTIONS (path 'file:///opt/data/nynta2020_26b', header 'true', inferSchema 'true');

DROP TABLE IF EXISTS nta;
CREATE TABLE nta
USING DELTA
AS SELECT * FROM auxi_nta_2020;


-- Analyzing shp file we found: EPSG:102718 
SELECT DISTINCT st_srid(geometry) FROM nta;

-- Nevertheless, the SRID automatically detected ESRI:102718  is not undertand by Sedona/GeooTools
-- Set the EPSG:2263 srid before the transformation, which is equivalet to EPSG:2263. 
-- Take into consideration that ST_Transform, in Sedona, requires the prefix EPSG

UPDATE nta SET geometry = ST_Transform(ST_SetSRID(geometry, 2263), 'EPSG:4326');


SELECT DISTINCT st_srid(geometry ) 
FROM nta;


>>> CREATE OR REPLACE TEMP VIEW auxi_nta_2020 USING shapefile
OPTIONS (path 'file://...
++
||
++
++


>>> DROP TABLE IF EXISTS nta...
++
||
++
++


>>> CREATE TABLE nta
USING DELTA
AS SELECT * FROM auxi_nta_2020...


++
||
++
++


>>> SELECT DISTINCT st_srid(geometry) FROM nta...


+-----------------+
|st_srid(geometry)|
+-----------------+
|102718           |
+-----------------+


>>> UPDATE nta SET geometry = ST_Transform(ST_SetSRID(geometry, 2263), 'EPSG:4326')...


26/08/22 22:57:15 WARN UpdateCommand: Could not validate number of records due to missing statistics.


+-----------------+
|num_affected_rows|
+-----------------+
|262              |
+-----------------+


>>> SELECT DISTINCT st_srid(geometry ) 
FROM nta...


+-----------------+
|st_srid(geometry)|
+-----------------+
|4326             |
+-----------------+



In [20]:
%%sql

DROP TABLE IF EXISTS stations;

CREATE TABLE stations (
  id STRING,
  name STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  geometry GEOMETRY,
  boroname STRING,
  ntacode STRING,
  ntaname STRING
)
USING DELTA;


-- Un solo escaneo de trips, consolidado y persistido
CREATE OR REPLACE TEMP VIEW stations_clean AS
WITH trip_starts AS (
  SELECT
    TRIM(CAST(start_station_id AS STRING)) AS id,
    start_station_name AS name,
    start_lat AS latitude,
    start_lng AS longitude
  FROM trips
  WHERE start_station_id IS NOT NULL

  UNION ALL

  SELECT
    TRIM(CAST(end_station_id AS STRING)) AS id,
    end_station_name AS name,
    end_lat AS latitude,
    end_lng AS longitude
  FROM trips
  WHERE end_station_id IS NOT NULL
)
SELECT
  id,
  FIRST(name, true) AS name,
  AVG(latitude) AS latitude,
  AVG(longitude) AS longitude
FROM trip_starts
GROUP BY id;


-- MERGE contra la vista ya calculada (liviana, sin recalcular)
MERGE INTO stations s
USING stations_clean n
ON s.id = n.id
WHEN MATCHED THEN UPDATE SET
  s.name = n.name,
  s.latitude = n.latitude,
  s.longitude = n.longitude,
  s.geometry = ST_Point(n.longitude, n.latitude)
WHEN NOT MATCHED THEN INSERT (
  id, name, latitude, longitude, geometry, boroname, ntacode, ntaname
)
VALUES (
  n.id, n.name, n.latitude, n.longitude,




  ST_Point(n.longitude, n.latitude),
  NULL, NULL, NULL
);


>>> DROP TABLE IF EXISTS stations...
++
||
++
++


>>> CREATE TABLE stations (
  id STRING,
  name STRING,
  latitude DOUBLE,
  longitu...
++
||
++
++


>>> CREATE OR REPLACE TEMP VIEW stations_clean AS
WITH trip_starts AS (
  SELECT
   ...
++
||
++
++


>>> MERGE INTO stations s
USING stations_clean n
ON s.id = n.id
WHEN MATCHED THEN UP...


+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|2358             |0               |0               |2358             |
+-----------------+----------------+----------------+-----------------+



26/08/22 23:02:15 WARN MapPartitionsRDD: RDD 291 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [21]:
%%sql

MERGE INTO stations s
USING nta n
ON ST_Within(s.geometry, n.geometry)
WHEN MATCHED THEN
  UPDATE SET 
    s.boroname = n.boroname,
    s.ntacode = n.nta2020,
    s.ntaname = n.ntaname;


>>> MERGE INTO stations s
USING nta n
ON ST_Within(s.geometry, n.geometry)
WHEN MATC...


+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|2273             |2273            |0               |0                |
+-----------------+----------------+----------------+-----------------+



In [22]:
%%sql

describe stations;

describe trips;


>>> describe stations...
+---------+---------+-------+
|col_name |data_type|comment|
+---------+---------+-------+
|id       |string   |NULL   |
|name     |string   |NULL   |
|latitude |double   |NULL   |
|longitude|double   |NULL   |
|geometry |geometry |NULL   |
|boroname |string   |NULL   |
|ntacode  |string   |NULL   |
|ntaname  |string   |NULL   |
+---------+---------+-------+


>>> describe trips...
+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|ride_id           |string   |NULL   |
|rideable_type     |string   |NULL   |
|started_at        |timestamp|NULL   |
|ended_at          |timestamp|NULL   |
|start_station_name|string   |NULL   |
|start_station_id  |string   |NULL   |
|end_station_name  |string   |NULL   |
|end_station_id    |string   |NULL   |
|start_lat         |double   |NULL   |
|start_lng         |double   |NULL   |
|end_lat           |double   |NULL   |
|end_lng           |double   |NULL   |
|memb